# 02 — Build Index
Chunks `scraped_corpus.json` using every strategy, tags metadata, embeds, and upserts to Qdrant.
Run once per chunking strategy. Results land in separate Qdrant collections:
`hr_rag_recursive`, `hr_rag_semantic`, `hr_rag_header`, `hr_rag_parent_child`.

In [ ]:
# ── 1. Clone repo & install deps ─────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/hr-wellness-rag.git
%cd hr-wellness-rag
!pip install -q -r requirements.txt

In [ ]:
# ── 2. Credentials ────────────────────────────────────────────────────────────
from google.colab import userdata, drive
drive.mount('/content/drive')

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
QDRANT_URL     = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

In [ ]:
# ── 3. Clients ────────────────────────────────────────────────────────────────
from openai import OpenAI
from qdrant_client import QdrantClient
from fastembed import SparseTextEmbedding

oai        = OpenAI(api_key=OPENAI_API_KEY)
qdrant     = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
bm25_model = SparseTextEmbedding(model_name='Qdrant/bm25')
print('Clients ready.')

In [ ]:
# ── 4. Load corpus ────────────────────────────────────────────────────────────
import json
import config

with open(config.CORPUS_PATH, encoding='utf-8') as f:
    corpus = json.load(f)

print(f'Corpus: {len(corpus)} documents')

In [ ]:
# ── 5. Chunk + index — run one cell per strategy ──────────────────────────────
# Change STRATEGY to: 'recursive', 'semantic', 'header', 'parent_child'

from pipeline.chunkers import get_chunks
from pipeline.embedder import build_index
import os

os.makedirs(config.CLEAN_CHUNKS_DIR, exist_ok=True)
os.makedirs(config.EMBEDDINGS_DIR,   exist_ok=True)

STRATEGY = 'recursive'   # ← change this

print(f'Chunking with strategy: {STRATEGY}')
chunks = get_chunks(STRATEGY, corpus, openai_api_key=OPENAI_API_KEY)
print(f'Chunks produced: {len(chunks):,}')

# Save chunks to Drive
chunks_path = f'{config.CLEAN_CHUNKS_DIR}/{STRATEGY}.json'
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print(f'Saved chunks to {chunks_path}')

# Tag, embed, upsert
build_index(
    chunks=chunks,
    strategy=STRATEGY,
    oai=oai,
    qdrant=qdrant,
    bm25_model=bm25_model,
    recreate=False,   # set True to wipe and rebuild the collection
)

In [ ]:
# ── 6. Verify ─────────────────────────────────────────────────────────────────
info = qdrant.get_collection(f'{config.COLLECTION_PREFIX}_{STRATEGY}')
print(f'Collection: hr_rag_{STRATEGY} — {info.points_count:,} points')